In [ ]:
!pip uninstall -y chronos --quiet
!pip install git+https://github.com/amazon-science/chronos-forecasting.git --quiet


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import requests
import pandas as pd
import numpy as np
import torch

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
import xgboost as xgb

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from chronos import ChronosPipeline

# =========================
# Helper
# =========================
def print_mse(nome, y_true, y_pred):
    print(f"{nome}: {mean_squared_error(y_true, y_pred):.2f}")

# =========================
# 📥 Download dataset
# =========================
url = 'https://raw.githubusercontent.com/antoinecarme/TimeSeriesData/master/fpp2/uschange.csv'
data = pd.read_csv(url)

# =========================
# 📊 Preprocessamento
# =========================
def float_to_quarterly_period(x):
    year = int(x)
    quarter = int(round((x - year) * 4)) + 1
    return pd.Period(f'{year}Q{quarter}', freq='Q')

data['Index'] = data['Index'].apply(float_to_quarterly_period)
data.set_index('Index', inplace=True)

# =========================
# 🔧 FEATURES (SEM LEAKAGE)
# =========================

for lag in range(1, 13):
    data[f'Consumption_lag{lag}'] = data['Consumption'].shift(lag)

data['diff_1'] = data['Consumption'].diff(1).shift(1)
data['diff_2'] = data['Consumption'].diff(2).shift(1)

data['rolling_mean_4'] = data['Consumption'].shift(1).rolling(4).mean()
data['rolling_std_4'] = data['Consumption'].shift(1).rolling(4).std()

data['trend'] = np.arange(len(data))

data['quarter'] = data.index.quarter
data['sin_q'] = np.sin(2 * np.pi * data['quarter'] / 4)
data['cos_q'] = np.cos(2 * np.pi * data['quarter'] / 4)

data['Income_lag1'] = data['Income'].shift(1)
data['Income_lag2'] = data['Income'].shift(2)
data['Unemployment_lag1'] = data['Unemployment'].shift(1)

data.dropna(inplace=True)

# =========================
# ✂️ Split
# =========================
train_size = int(len(data) * 0.8)
train = data.iloc[:train_size]
test = data.iloc[train_size:]

feature_cols = [col for col in data.columns if col != 'Consumption']

X_train = train[feature_cols]
y_train = train['Consumption']

X_test = test[feature_cols]
y_test = test['Consumption']

# =========================
# 🚀 MODELOS
# =========================

# Linear
model_lr = LinearRegression().fit(X_train, y_train)
print_mse("Linear", y_test, model_lr.predict(X_test))

# Ridge
model_ridge = Ridge(alpha=1.0).fit(X_train, y_train)
print_mse("Ridge", y_test, model_ridge.predict(X_test))

# ARIMA
try:
    pred_arima = ARIMA(y_train, order=(1,1,1)).fit().forecast(len(y_test))
    print_mse("ARIMA", y_test, pred_arima)
except:
    print("Erro ARIMA")

# SARIMAX
try:
    pred_sarimax = SARIMAX(y_train, order=(1,1,1), seasonal_order=(1,1,1,4)).fit().forecast(len(y_test))
    print_mse("SARIMAX", y_test, pred_sarimax)
except:
    print("Erro SARIMAX")

# XGBoost
model_xgb = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
).fit(X_train, y_train)

print_mse("XGB", y_test, model_xgb.predict(X_test))

# RF
model_rf = RandomForestRegressor(random_state=42).fit(X_train, y_train)
print_mse("RF", y_test, model_rf.predict(X_test))

# GB
model_gb = GradientBoostingRegressor(random_state=42).fit(X_train, y_train)
print_mse("GB", y_test, model_gb.predict(X_test))

# Stacking
model_stack = StackingRegressor(
    estimators=[
        ('lr', LinearRegression()),
        ('xgb', xgb.XGBRegressor(n_estimators=200, max_depth=3)),
        ('rf', RandomForestRegressor(random_state=42))
    ],
    final_estimator=Ridge(alpha=1.0)
).fit(X_train, y_train)

print_mse("Stacking", y_test, model_stack.predict(X_test))

# =========================
# 🧠 CHRONOS (CORRETO)
# =========================
try:
    pipeline = ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-small",
        device_map="cpu",
        dtype=torch.float32
    )

    # usa apenas contexto recente
    y_context = y_train.tail(60)

    # normalização robusta
    median = y_context.median()
    iqr = y_context.quantile(0.75) - y_context.quantile(0.25)

    context = (y_context - median) / iqr
    context = torch.tensor(context.values, dtype=torch.float32).flatten()

    prediction_length = len(y_test)

    forecast = pipeline.predict(
        context,
        prediction_length=prediction_length,
        num_samples=100
    )

    forecast = np.array(forecast)


    # 🔥 CORREÇÃO PRINCIPAL
    forecast = np.median(forecast, axis=1).flatten()

    # desnormalizar
    forecast = forecast * iqr + median

    pred_chronos = pd.Series(forecast, index=y_test.index)

    print_mse("Chronos", y_test, pred_chronos)

except Exception as e:
    print("Erro Chronos:", e)

Linear: 0.07
Ridge: 0.07
ARIMA: 0.25
SARIMAX: 0.26
XGB: 0.08
RF: 0.11
GB: 0.10
Stacking: 0.06
Shape Chronos: (1, 100, 35)
Chronos: 0.43
